# Prepare Synthetic Mag Survey Data

Generates a synthetic airborne magnetic survey over a known prismatic
susceptibility body and saves the result as a real `MagData` file that
the main inversion notebook can load.

**Run this notebook once** before running `mag_inversion_example.ipynb`.

## What it does

1. Defines a survey grid (E–W flight lines at 100 m spacing, 60 m AGL).
2. Builds a TensorMesh with a rectangular prism of χ = 0.1 SI at 100–200 m depth.
3. Forward-models TMI using `SimPEG.potential_fields.magnetics`.
4. Adds realistic Gaussian noise.
5. Packages as an `AirMagTools.MagData` object with the correct column names
   and index (`line`, `fidcount`) and saves to `synthetic_survey.msgpack`.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from SimPEG import maps
from SimPEG.potential_fields import magnetics
from discretize import TensorMesh

from AirMagTools.magdata import MagData

## 1  Survey geometry

In [ ]:
# CRS: a UTM zone (EPSG:32632 — WGS84 / UTM zone 32N).
# Coordinates are local offsets; origin is an arbitrary survey datum.
ORIGIN_EASTING  = 500_000.0   # m  (UTM easting of survey origin)
ORIGIN_NORTHING = 6_500_000.0 # m  (UTM northing of survey origin)

FLIGHT_ALT  = 60.0    # m AGL (above ground, flat terrain assumed)
LINE_SPACE  = 100.0   # m between flight lines
SOUNDING_DX = 20.0    # m along-line sampling interval
SURVEY_HALF = 1000.0  # m half-width of survey square

# Earth field parameters (Northern Europe)
FIELD_INTENSITY  = 52_000.0  # nT
FIELD_INCL       = 65.0      # degrees
FIELD_DECL       = 5.0       # degrees

# Build flight lines (E–W)
y_lines = np.arange(-SURVEY_HALF, SURVEY_HALF + LINE_SPACE, LINE_SPACE)
x_snd   = np.arange(-SURVEY_HALF, SURVEY_HALF + SOUNDING_DX, SOUNDING_DX)

rows = []
for line_num, y_line in enumerate(y_lines):
    for fid, x in enumerate(x_snd):
        rows.append({
            'line':     int(line_num * 1000),  # conventional: line 0, 1000, 2000 …
            'fidcount': fid,
            'easting':  ORIGIN_EASTING  + x,
            'northing': ORIGIN_NORTHING + y_line,
            'gpsalt':   FLIGHT_ALT,
        })

df_geom = pd.DataFrame(rows)
print(f"{len(df_geom)} soundings over {len(y_lines)} lines")

## 2  True susceptibility model

In [ ]:
# Dense TensorMesh for forward modelling — use local coordinates (origin at 0)
cs = 25.0   # forward model cell size (m)
n_pad = 10  # padding cells each side

nx = int(2 * SURVEY_HALF / cs) + 2 * n_pad
ny = nx
nz = 20    # 500 m depth range

mesh_true = TensorMesh(
    [np.ones(nx) * cs,
     np.ones(ny) * cs,
     np.ones(nz) * cs],
    origin=[-SURVEY_HALF - n_pad * cs,
            -SURVEY_HALF - n_pad * cs,
            -nz * cs]  # z = –500 m … 0 m
)

cc = mesh_true.cell_centers
chi_true = np.zeros(mesh_true.nC)

# Rectangular prism: 400 × 400 m footprint, 100–200 m depth, χ = 0.1 SI
prism = (
    (np.abs(cc[:, 0]) < 200) &
    (np.abs(cc[:, 1]) < 200) &
    (cc[:, 2] > -200) &
    (cc[:, 2] < -100)
)
chi_true[prism] = 0.1

print(f"True mesh: {mesh_true.nC} cells  |  prism cells: {prism.sum()}")

fig, ax = plt.subplots(figsize=(5, 4))
slice_idx = nz // 2 - 2  # ~150 m depth
chi_slice = chi_true.reshape(nx, ny, nz)[:, :, slice_idx]
im = ax.imshow(chi_slice.T, origin='lower',
               extent=[-SURVEY_HALF - n_pad*cs, SURVEY_HALF + n_pad*cs,
                       -SURVEY_HALF - n_pad*cs, SURVEY_HALF + n_pad*cs],
               cmap='viridis')
plt.colorbar(im, ax=ax, label='χ (SI)')
ax.set_title('True model — horizontal slice at z ≈ –150 m')
ax.set_xlabel('Easting offset (m)'); ax.set_ylabel('Northing offset (m)')
plt.tight_layout(); plt.show()

## 3  Forward model TMI

In [ ]:
# Receiver locations in local coords (mesh origin at 0,0)
x_local = df_geom.easting.values  - ORIGIN_EASTING
y_local = df_geom.northing.values - ORIGIN_NORTHING
z_local = df_geom.gpsalt.values
receiver_locs = np.c_[x_local, y_local, z_local]

receiver = magnetics.receivers.Point(receiver_locs, components=['tmi'])
source   = magnetics.sources.SourceField(
    receiver_list=[receiver],
    parameters=[FIELD_INTENSITY, FIELD_INCL, FIELD_DECL]
)
survey_fwd = magnetics.survey.Survey(source)

sim_fwd = magnetics.simulation.Simulation3DIntegral(
    mesh=mesh_true,
    survey=survey_fwd,
    chiMap=maps.IdentityMap(nP=mesh_true.nC),
    store_sensitivities='forward_only',
)

tmi_clean = sim_fwd.dpred(chi_true)
print(f"Peak anomaly: {tmi_clean.max():.1f} nT  |  min: {tmi_clean.min():.1f} nT")

In [ ]:
# Add Gaussian noise: 1 % fractional + 1 nT floor
np.random.seed(42)
noise_std  = 0.01 * np.abs(tmi_clean) + 1.0
tmi_noisy  = tmi_clean + noise_std * np.random.randn(len(tmi_clean))

plt.figure(figsize=(8, 4))
sc = plt.scatter(x_local, y_local, c=tmi_noisy, s=2, cmap='RdBu_r')
plt.colorbar(sc, label='TMI (nT)')
plt.title('Synthetic observed TMI')
plt.xlabel('Easting offset (m)'); plt.ylabel('Northing offset (m)')
plt.axis('equal'); plt.tight_layout(); plt.show()

## 4  Package as MagData and save

The `MagData` class expects a DataFrame indexed by `(line, fidcount)`.  Column
names follow the AirMagTools normalisation standard.  The compensated field is
`magcom`; we also store `maguncom` (identical here — no compensation applied to
synthetic data).  A synthetic `utctime` is added because some MagData methods
expect it.

In [ ]:
# Build the DataFrame
df = df_geom.copy()
df['magcom']   = tmi_noisy
df['maguncom'] = tmi_noisy          # no compensation for synthetic data
df['diurnal']  = 0.0                # no diurnal correction
df['residual'] = tmi_noisy - tmi_noisy.mean()

# Synthetic UTC time (1 sample per second along each line)
t0 = pd.Timestamp('2024-06-15 08:00:00', tz='UTC').timestamp()
df['utctime'] = t0 + df.groupby('line').cumcount().values

df = df.set_index(['line', 'fidcount'])

mag_data = MagData(
    df,
    crs='EPSG:32632',
    field_intensity=FIELD_INTENSITY,
    field_inclination=FIELD_INCL,
    field_declination=FIELD_DECL,
    survey_name='synthetic_prism',
)

print(mag_data)

In [ ]:
out_path = 'synthetic_survey.msgpack'
mag_data.save(out_path)
print(f"Saved to {out_path}")

The file `synthetic_survey.msgpack` is now ready.  Open `mag_inversion_example.ipynb` to run the inversion.